In [11]:
# Import packages
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [12]:
# Load in dataset
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/subject-info.csv")

df = pd.read_csv(
    path,
    sep=";",          # correct delimiter
    decimal=",",      # European decimal format
    engine="python",  # handle irregular formatting
    na_values=["", "NA"] # handles missing values
)

# Clean (tabs inside numbers)
df = df.replace(r"\t", ".", regex=True)

# Convert age to numeric
df["Age"] = (
    df["Age"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

print(df.shape) # 992 patients with 103 columns

df.head()

(992, 103)


,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,Age,Gender (male=1),Weight (kg),Height (cm),Body Mass Index (Kg/m2),...,Angiotensin-II receptor blocker (yes=1),Anticoagulants/antitrombotics (yes=1),Betablockers (yes=1),Digoxin (yes=1),Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1)
0,P0001,2065,1460,NaN,0,58.0,1,83,163,31.2,...,0,1,1,1,1,0,0,0,1,0
1,P0002,2045,1460,NaN,0,58.0,1,74,160,28.9,...,1,1,1,0,0,0,1,0,0,0
2,P0003,2044,1460,NaN,0,69.0,1,83,174,27.4,...,1,1,1,1,1,0,0,0,0,0
3,P0004,2044,1460,NaN,0,56.0,0,84,165,30.9,...,1,1,1,0,1,1,0,0,0,0
4,P0005,2043,1460,NaN,0,70.0,1,97,183,29.0,...,0,1,1,0,1,0,1,0,1,1


In [13]:
df["Patient ID"] = df["Patient ID"].str.replace("P", "", regex=False)
df["Patient ID"] = df["Patient ID"].str.strip()

In [14]:
# Patient selection
mask = (
    (df["Holter available"] != 0) &  # Select patients with Holter
    (df["Prior implantable device"] == 0) &  # Remove pacemaker patients
    ((df["Exit of the study"].ne(2)) | (df["Exit of the study"].isna()))  # Remove cardiac transplant
)

df2 = df[mask].copy()

# Exit of study is 0 for survivor
df2["Exit of the study"] = df2["Exit of the study"].fillna(0).astype(int)

# Combine Cause of death codes:
# 6 and 7 -> Pump failure death
df2.loc[df2["Cause of death"].isin([6, 7]), "Cause of death"] = 6

# Time to event
df2["time_to_event_days"] = df2[
    ["Follow-up period from enrollment (days)", "days_4years"]
].min(axis=1)

# Any cardiac death (SCD or PFD)
df2["event_cardiac"] = df2["Cause of death"].isin([3, 6]).astype(int)

# Cause-specific events
df2["event_cardiac_SCD"] = (df2["Cause of death"] == 3).astype(int)
df2["event_cardiac_PFD"] = (df2["Cause of death"] == 6).astype(int)

print("Cause of death counts:")
print(df2["Cause of death"].value_counts().sort_index())

print("\nEvent counts:")
print(df2[["event_cardiac", "event_cardiac_SCD", "event_cardiac_PFD"]].sum())

# Table of number of patients per outcome
cause_counts = df2["Cause of death"].value_counts().sort_index()
print(cause_counts)

# Note: Original paper reports 94/996 SCDs (9.4%) and 111/996 PFDs (11.1%)
# Final counts align very well with 74/796 SCDs (9.3%) and 87/796 PFDs (10.9%)

Cause of death counts:
Cause of death
0    585
1     50
3     74
6     87
Name: count, dtype: int64

Event counts:
event_cardiac        161
event_cardiac_SCD     74
event_cardiac_PFD     87
dtype: int64
Cause of death
0    585
1     50
3     74
6     87
Name: count, dtype: int64


In [15]:
# --------------------------------------------------
# Select survival-relevant label columns
# NOTE: requires TRUE follow-up column to be present
# --------------------------------------------------
labels = df2[
    [
        "Patient ID",
        "time_to_event_days",                 # truncated (for training)
        "event_cardiac",
        "event_cardiac_SCD",
        "event_cardiac_PFD",
        "Follow-up period from enrollment (days)"  # TRUE follow-up
    ]
].copy()

labels = labels.reset_index(drop=True)

# --------------------------------------------------
# Create follow-up time bins using TRUE follow-up
# (important for time-dependent AUC stability)
# --------------------------------------------------
max_fu = labels["Follow-up period from enrollment (days)"].max()

labels["followup_bin"] = pd.cut(
    labels["Follow-up period from enrollment (days)"],
    bins=[0, 365, 730, 1095, max_fu + 1],
    labels=["<1y", "1–2y", "2–3y", "≥3y"],
    include_lowest=True,
)

# --------------------------------------------------
# Create stratification variable:
#   cardiac event (yes/no) × follow-up bin
# --------------------------------------------------
labels["strata"] = (
    labels["event_cardiac"].astype(str) + "_" +
    labels["followup_bin"].astype(str)
)

# Optional: inspect strata counts
print("\nStrata counts (event × follow-up bin):")
print(labels["strata"].value_counts().sort_index())

# --------------------------------------------------
# Stratified K-fold split using combined strata
# --------------------------------------------------
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

labels["fold"] = -1

for fold, (_, val_idx) in enumerate(
    skf.split(labels["Patient ID"], labels["strata"])
):
    labels.loc[val_idx, "fold"] = fold

# --------------------------------------------------
# Sanity checks
# --------------------------------------------------
print("\nPatients per fold:")
print(labels["fold"].value_counts().sort_index())

print("\nCardiac event counts per fold:")
print(
    labels.groupby(["fold", "event_cardiac"])
          .size()
          .unstack(fill_value=0)
)

print("\nFollow-up bins per fold (TRUE follow-up):")
print(
    labels.groupby(["fold", "followup_bin"])
          .size()
          .unstack(fill_value=0)
)

print("\nSCD / PFD breakdown per fold:")
print(
    labels.groupby(["fold", "event_cardiac_SCD", "event_cardiac_PFD"])
          .size()
)


Strata counts (event × follow-up bin):
strata
0_1–2y     18
0_2–3y     16
0_<1y      14
0_≥3y     587
1_1–2y     35
1_2–3y     39
1_<1y      58
1_≥3y      29
Name: count, dtype: int64

Patients per fold:
fold
0    160
1    159
2    159
3    159
4    159
Name: count, dtype: int64

Cardiac event counts per fold:
event_cardiac    0   1
fold                  
0              128  32
1              127  32
2              126  33
3              126  33
4              128  31

Follow-up bins per fold (TRUE follow-up):
followup_bin  <1y  1–2y  2–3y  ≥3y
fold                              
0              14    11    11  124
1              14    10    11  124
2              15    10    11  123
3              14    11    11  123
4              15    11    11  122

SCD / PFD breakdown per fold:
fold  event_cardiac_SCD  event_cardiac_PFD
0     0                  0                    128
                         1                     13
      1                  0                     19
1     0       

/tmp/ipykernel_934293/3345719772.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  labels.groupby(["fold", "followup_bin"])


In [16]:
# Save labels
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1")

# --------------------------------------------------
# 1) Overall cardiac death (SCD + PFD)
# --------------------------------------------------
labels_cardiac = labels[
    ["Patient ID", "time_to_event_days", "event_cardiac", "fold", "Follow-up period from enrollment (days)"]
].copy()

out_file_cardiac = path / "music_patient_folds_5cv_survival_cardiac_time.csv"
labels_cardiac.to_csv(out_file_cardiac, index=False)

# --------------------------------------------------
# 2) Sudden cardiac death (cause-specific)
#     Rename event_cardiac_SCD -> event_cardiac
# --------------------------------------------------
labels_scd = labels[
    ["Patient ID", "time_to_event_days", "event_cardiac_SCD", "fold", "Follow-up period from enrollment (days)"]
].copy()

labels_scd = labels_scd.rename(
    columns={"event_cardiac_SCD": "event_cardiac"}
)

out_file_scd = path / "music_patient_folds_5cv_survival_SCD_time.csv"
labels_scd.to_csv(out_file_scd, index=False)

# --------------------------------------------------
# 3) Pump failure death (cause-specific)
#     Rename event_cardiac_PFD -> event_cardiac
# --------------------------------------------------
labels_pfd = labels[
    ["Patient ID", "time_to_event_days", "event_cardiac_PFD", "fold", "Follow-up period from enrollment (days)"]
].copy()

labels_pfd = labels_pfd.rename(
    columns={"event_cardiac_PFD": "event_cardiac"}
)

out_file_pfd = path / "music_patient_folds_5cv_survival_PFD_time.csv"
labels_pfd.to_csv(out_file_pfd, index=False)

print("Saved survival label CSVs:")
print(out_file_cardiac)
print(out_file_scd)
print(out_file_pfd)

Saved survival label CSVs:
../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/music_patient_folds_5cv_survival_cardiac_time.csv
../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/music_patient_folds_5cv_survival_SCD_time.csv
../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/music_patient_folds_5cv_survival_PFD_time.csv
